In [1]:
from mmpose.apis import MMPoseInferencer
from datetime import datetime
import subprocess
import os
import glob

while os.getcwd().split('/')[-1] != 'mmpose-synthetic-tune':
    os.chdir('..')

In [2]:
class MMPoseModelCoach:
    command = 'python'
    script = 'mmpose/tools/train.py'
    
    detector_model = {  #rtmdet
        "det_model": 'mmdetection/configs/rtmdet/rtmdet_l_swin_b_p6_4xb16-100e_coco.py',
        "det_weights": 'checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth'
    }

    def __init__(self, config=None, resume=True, work_dir=None, notes=''):
        current_time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

        # self.config = config
        self.resume = '--resume' if resume == True else ''
        if work_dir is not None:
            self.work_dir = work_dir
            self.config_path = glob.glob(f'{self.work_dir}/*.py')[0]
        else:
            self.config_path = f'dataset-coco/custom-configs/{self.config}'
            self.work_dir = f'models/_train-{current_time}-{notes}'

        self.args = [
            self.command,
            self.script,
            self.config_path,
            '--work-dir',
            self.work_dir,
            self.resume,
        ]

    def train(self):
        subprocess.run(self.args)

    def visualize_results(self, vis_input, model_ckpt=None, radius=4, thickness=1):
        if model_ckpt is None:
            checkpoint_path = glob.glob(f'{self.work_dir}/best*.pth')[0]
        else:
            checkpoint_path = f'{self.work_dir}/{model_ckpt}'
        
        poser_model = {
            "pose2d": self.config_path,
            "pose2d_weights": checkpoint_path
        }

        inferencer = MMPoseInferencer(**poser_model, **self.detector_model, device='cuda:0')

        input_path = vis_input
        output_path = f'{self.work_dir}/vis_results'

        result_generator = inferencer(
            input_path,
            radius=radius,
            thickness=thickness,
            vis_out_dir=output_path,
            draw_heatmap=True,
            det_cat_ids=5
        )

        results = [res for res in result_generator]


### Train and Visualize

In [7]:
_20kp = MMPoseModelCoach(
    config='td-hm_hrnet-w48_8xb64-210e_ap10k-256x256-finetune-20kp.py',
    notes='20kp-train-on-ap10k-w48'
)

base_hrnet.train()

_20kp.visualize_results(
    vis_input='/home/galiold/projects/datasets/frames/8/8_img00045.png',
)

04/23 12:58:44 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.9.19 (main, Apr  6 2024, 17:57:55) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1764525799
    GPU 0: NVIDIA GeForce RTX 4070
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.8, V11.8.89
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.2.1+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.3.2 (Git Hash 2dc95a2ad0841e29db8b22fbccaf3e5da7992b01)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compu

### Visualize Only

In [10]:
vis_input = '/home/galiold/Pictures/Screenshots/test-29.png'

In [11]:
_20kp_on_ap10k = MMPoseModelCoach(
    config='td-hm_hrnet-w48_8xb64-210e_ap10k-256x256-finetune-20kp.py',
    work_dir='models/_train-2024-04-23_12-58-41-20kp-train-on-ap10k-w48'
)
_20kp_on_ap10k.visualize_results(
    vis_input=vis_input
)

Loads checkpoint by local backend from path: models/_train-2024-04-23_12-58-41-20kp-train-on-ap10k-w48/best_coco_AP_epoch_180.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
04/23 15:11:54 - mmengine - INFO - the output image has been saved at models/_train-2024-04-23_12-58-41-20kp-train-on-ap10k-w48/vis_results/test-29.png


In [12]:
pretrained_ap10k = MMPoseModelCoach(
    work_dir='models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029'
)
pretrained_ap10k.visualize_results(
    model_ckpt='hrnet_w48_ap10k_256x256-d95ab412_20211029.pth',
    vis_input=vis_input
)

Loads checkpoint by local backend from path: models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/hrnet_w48_ap10k_256x256-d95ab412_20211029.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
04/23 15:11:58 - mmengine - INFO - the output image has been saved at models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/vis_results/test-29.png
